In [1]:
import os
 
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
 
sns.set_theme(style="whitegrid")
os.makedirs("charts", exist_ok=True)

In [2]:
df = pd.read_csv("customer_shopping_behavior_clean.csv")

In [3]:
print("Shape:", df.shape)


Shape: (3900, 18)


In [6]:

print("\nFirst 5 rows:\n", df.head())
print("\nInfo:")
df.info()





First 5 rows:
    customer_id  age gender item_purchased  category  purchase_amount_usd  \
0            1   55   Male         Blouse  Clothing                   53   
1            2   19   Male        Sweater  Clothing                   64   
2            3   50   Male          Jeans  Clothing                   73   
3            4   21   Male        Sandals  Footwear                   90   
4            5   45   Male         Blouse  Clothing                   49   

        location size      color  season  review_rating subscription_status  \
0       Kentucky    L       Gray  Winter            3.1                 Yes   
1          Maine    L     Maroon  Winter            3.1                 Yes   
2  Massachusetts    S     Maroon  Spring            3.1                 Yes   
3   Rhode Island    M     Maroon  Spring            3.5                 Yes   
4         Oregon    M  Turquoise  Spring            2.7                 Yes   

   shipping_type discount_applied promo_code_used  p

In [7]:

print("\nSummary statistics:\n", df.describe())



Summary statistics:
        customer_id          age  purchase_amount_usd  review_rating  \
count  3900.000000  3900.000000          3900.000000    3863.000000   
mean   1950.500000    44.068462            59.764359       3.750065   
std    1125.977353    15.207589            23.685392       0.716983   
min       1.000000    18.000000            20.000000       2.500000   
25%     975.750000    31.000000            39.000000       3.100000   
50%    1950.500000    44.000000            60.000000       3.800000   
75%    2925.250000    57.000000            81.000000       4.400000   
max    3900.000000    70.000000           100.000000       5.000000   

       previous_purchases  
count         3900.000000  
mean            25.351538  
std             14.447125  
min              1.000000  
25%             13.000000  
50%             25.000000  
75%             38.000000  
max             50.000000  


In [8]:
print("\nMissing values:\n", df.isnull().sum())

 


Missing values:
 customer_id                0
age                        0
gender                     0
item_purchased             0
category                   0
purchase_amount_usd        0
location                   0
size                       0
color                      0
season                     0
review_rating             37
subscription_status        0
shipping_type              0
discount_applied           0
promo_code_used            0
previous_purchases         0
payment_method             0
frequency_of_purchases     0
dtype: int64


In [9]:
print("\nDuplicate rows:", df.duplicated().sum())


Duplicate rows: 0


In [10]:
# Fill missing review ratings with the median rating of each product category
df["review_rating"] = df.groupby("category")["review_rating"].transform(
    lambda s: s.fillna(s.median())
)

In [11]:
# Yes/No -> True/False
for col in ["subscription_status", "discount_applied", "promo_code_used"]:
    df[col] = df[col].map({"Yes": True, "No": False})
 

In [12]:
# Age groups
df["age_group"] = pd.cut(
    df["age"],
    bins=[17, 25, 35, 45, 55, 65, 120],
    labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66+"],
)
 

In [13]:
# Customer segments based on previous purchases
df["customer_segment"] = pd.cut(
    df["previous_purchases"],
    bins=[0, 1, 10, df["previous_purchases"].max()],
    labels=["New", "Returning", "Loyal"],
)

In [14]:
print("\nMissing values after cleaning:", df["review_rating"].isnull().sum())


Missing values after cleaning: 0


In [18]:
 
# Q1. Revenue by gender
print("\nRevenue by gender:")
print(df.groupby("gender")["purchase_amount_usd"].sum())


Revenue by gender:
gender
Female     75191
Male      157890
Name: purchase_amount_usd, dtype: int64


In [19]:
# Q2. Discount users who spent more than the average purchase
avg_purchase = df["purchase_amount_usd"].mean()
high_spend_discount = df[(df["discount_applied"]) & (df["purchase_amount_usd"] >= avg_purchase)]
print(f"\nDiscount users spending >= average (${avg_purchase:.2f}):", len(high_spend_discount))


Discount users spending >= average ($59.76): 839


In [20]:
# Q3. Top 5 products by average review rating
top_rated = (
    df.groupby("item_purchased")["review_rating"].mean().round(2)
    .sort_values(ascending=False).head(5)
)
print("\nTop 5 rated products:\n", top_rated)


Top 5 rated products:
 item_purchased
Gloves     3.86
Sandals    3.84
Boots      3.82
Hat        3.80
T-shirt    3.78
Name: review_rating, dtype: float64


In [21]:
# Q4. Standard vs Express shipping
print("\nAvg purchase by shipping type:")
print(
    df[df["shipping_type"].isin(["Standard", "Express"])]
    .groupby("shipping_type")["purchase_amount_usd"].mean().round(2)
)
 


Avg purchase by shipping type:
shipping_type
Express     60.48
Standard    58.46
Name: purchase_amount_usd, dtype: float64


In [22]:
# Q5. Subscribers vs non-subscribers
print("\nSubscribers vs non-subscribers:")
print(
    df.groupby("subscription_status")["purchase_amount_usd"]
    .agg(customers="count", avg_spend="mean", total_revenue="sum").round(2)
)
 


Subscribers vs non-subscribers:
                     customers  avg_spend  total_revenue
subscription_status                                     
False                     2847      59.87         170436
True                      1053      59.49          62645


In [23]:
# Q6. Top 5 products by discount rate
discount_rate = (
    df.groupby("item_purchased")["discount_applied"].mean().mul(100).round(2)
    .sort_values(ascending=False).head(5)
)
print("\nTop 5 products by discount rate (%):\n", discount_rate)
 


Top 5 products by discount rate (%):
 item_purchased
Hat         50.00
Sneakers    49.66
Coat        49.07
Sweater     48.17
Pants       47.37
Name: discount_applied, dtype: float64


In [24]:
# Q7. Customer segments
segment_counts = df["customer_segment"].value_counts()
print("\nCustomer segments:\n", segment_counts)


Customer segments:
 customer_segment
Loyal        3116
Returning     701
New            83
Name: count, dtype: int64


In [25]:
# Q8. Top 3 products within each category
top3 = (
    df.groupby(["category", "item_purchased"]).size().reset_index(name="total_orders")
    .sort_values(["category", "total_orders"], ascending=[True, False])
    .groupby("category").head(3)
)
print("\nTop 3 products per category:\n", top3)


Top 3 products per category:
        category item_purchased  total_orders
5   Accessories        Jewelry           171
1   Accessories           Belt           161
7   Accessories     Sunglasses           161
8      Clothing         Blouse           171
12     Clothing          Pants           171
13     Clothing          Shirt           169
20     Footwear        Sandals           160
21     Footwear          Shoes           150
22     Footwear       Sneakers           145
24    Outerwear         Jacket           163
23    Outerwear           Coat           161


In [26]:
 
# Q9. Do repeat buyers subscribe?
repeat = df[df["previous_purchases"] > 5]
print("\nRepeat buyers by subscription:\n", repeat["subscription_status"].value_counts())


Repeat buyers by subscription:
 subscription_status
False    2518
True      958
Name: count, dtype: int64


In [27]:
# Q10. Revenue by age group
age_revenue = (
    df.groupby("age_group", observed=True)["purchase_amount_usd"].sum()
    .sort_values(ascending=False)
)
print("\nRevenue by age group:\n", age_revenue)
 


Revenue by age group:
 age_group
46-55    45619
56-65    44352
26-35    44342
36-45    43234
18-25    34630
66+      20904
Name: purchase_amount_usd, dtype: int64
